In [1]:
import os
import time
import re
import json
import pandas as pd
from concurrent.futures import ThreadPoolExecutor, as_completed

from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.common.keys import Keys
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from selenium.common.exceptions import (
    ElementClickInterceptedException,
    StaleElementReferenceException,
    TimeoutException,
)


def safe_click_elem(driver, elem, retries=6):
    for _ in range(retries):
        try:
            driver.execute_script("arguments[0].scrollIntoView({block:'center'});", elem)
            time.sleep(0.15)
            try:
                elem.click()
            except (ElementClickInterceptedException, StaleElementReferenceException):
                driver.execute_script("arguments[0].click();", elem)
            return True
        except Exception:
            time.sleep(0.4)
    return False


def close_popup(driver):
    try:
        btn = WebDriverWait(driver, 2).until(
            EC.element_to_be_clickable((By.CSS_SELECTOR, "button[class*='Modal_close']"))
        )
        safe_click_elem(driver, btn)
        return True
    except Exception:
        return False


def _clean_text(s):
    return " ".join((s or "").split()).strip()


def _make_driver():
    opts = Options()
    opts.add_argument("--headless=new")
    opts.add_argument("--window-size=1280,800")
    opts.add_argument("--no-sandbox")
    opts.add_argument("--disable-dev-shm-usage")

    opts.add_experimental_option("prefs", {
        "profile.managed_default_content_settings.images": 2,
    })

    opts.add_argument(
        "user-agent=Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
        "AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36"
    )
    return webdriver.Chrome(options=opts)


def _dedup_keep_order(items):
    out, seen = [], set()
    for x in items:
        x = _clean_text(x)
        if not x:
            continue
        if x not in seen:
            seen.add(x)
            out.append(x)
    return out


def _split_welfare(raw: str):
    if not raw:
        return []
    raw = raw.replace("\u00a0", " ")
    raw = raw.replace('"', "").replace("'", "")
    raw = raw.replace("·", " ")
    raw = re.sub(r"\s+", " ", raw).strip()

    parts = []
    for chunk in re.split(r"[,\n]", raw):
        t = chunk.strip()
        if t:
            parts.append(t)

    return _dedup_keep_order(parts)


def fetch_detail(link_no_scheme):
    url = "https://" + link_no_scheme
    company = ""
    detail_list = []

    driver = None
    try:
        driver = _make_driver()
        driver.get(url)

        try:
            WebDriverWait(driver, 6).until(
                EC.presence_of_element_located((By.CSS_SELECTOR, "h2, #cntrName, .company_name"))
            )
        except Exception:
            time.sleep(1)

        close_popup(driver)

        for sel in ["h2", ".company_name", "#cntrName", "div.company_name", "span.company_name"]:
            try:
                t = _clean_text(driver.find_element(By.CSS_SELECTOR, sel).text)
                if t and len(t) <= 60:
                    company = t
                    break
            except Exception:
                pass

        welfare_items = []
        try:
            welfare_block = driver.find_element(
                By.XPATH,
                "//div[@data-sentry-element='Flex' and .//span[contains(normalize-space(.),'복리후생')]]"
            )
            spans = welfare_block.find_elements(By.XPATH, ".//span[@data-sentry-element='Typography']")

            welfare_texts = []
            for sp in spans:
                t = (sp.get_attribute("innerText") or sp.text or "").strip()
                if not t:
                    continue
                if "복리후생" in t and len(t) <= 10:
                    continue
                welfare_texts.append(t)

            joined = "\n".join(welfare_texts).strip()
            welfare_items = _split_welfare(joined)
        except Exception:
            welfare_items = []

        kws = [
            "경력", "신입", "학력무관", "대졸", "석사", "박사", "고졸",
            "정규직", "계약직", "인턴", "프리랜서", "파견직", "아르바이트",
            "서울", "경기", "인천", "부산", "대구", "광주", "대전", "울산",
            "세종", "강원", "충북", "충남", "전북", "전남", "경북", "경남", "제주",
            "D-", "#"
        ]
        other_items = []
        seen = set()
        for el in driver.find_elements(By.CSS_SELECTOR, "dd, li, span, td"):
            t = _clean_text(el.get_attribute("innerText") or el.text)
            if not t or t in seen:
                continue

            if len(t) > 140:
                continue

            if any(kw in t for kw in kws):
                seen.add(t)
                other_items.append(t)

            if len(other_items) >= 12:
                break

        detail_list = _dedup_keep_order(welfare_items + other_items)

    except Exception:
        pass
    finally:
        if driver:
            try:
                driver.quit()
            except Exception:
                pass

    return company, detail_list


def collect_cards(driver, limit=50):
    cards = driver.find_elements(
        By.XPATH,
        "//div[contains(@class,'styles_mb_space') and .//a[contains(@href,'GI_Read')]]"
    )
    if not cards:
        cards = driver.find_elements(
            By.XPATH,
            "//div[@data-sentry-element='Block' and .//a[contains(@href,'GI_Read')]]"
        )
    dedup, seen = [], set()
    for c in cards:
        try:
            a = c.find_element(By.CSS_SELECTOR, "a[data-sentry-element='BaseLink'][href*='GI_Read']")
            href = (a.get_attribute("href") or "").strip()
            if href and href not in seen:
                seen.add(href)
                dedup.append(c)
        except Exception:
            pass
    return dedup[:limit]


def extract_title_link(card):
    title, link = "", ""
    for a in card.find_elements(By.CSS_SELECTOR, "a[data-sentry-element='BaseLink'][href*='GI_Read']"):
        spans = a.find_elements(By.CSS_SELECTOR, "span[data-sentry-element='Typography']")
        g700, g500, g900, others = [], [], [], []
        for sp in spans:
            color = sp.get_attribute("data-accent-color") or ""
            t = _clean_text(sp.text)
            if not t:
                continue
            if color == "gray700":
                g700.append(t)
            elif color == "gray500":
                g500.append(t)
            elif color == "gray900":
                g900.append(t)
            else:
                others.append(t)

        if g700:
            title = (g500 or g900 or others or [""])[0]
        elif g900:
            title = g900[0]
        elif others:
            title = max(others, key=len)

        href = (a.get_attribute("href") or "").strip()
        if href.startswith("/"):
            href = "https://www.jobkorea.co.kr" + href
        link = href.replace("https://", "").replace("http://", "")
        break
    return title, link


def crawl_jobkorea_onepage_to_csv(
    keyword="데이터분석",
    out_path="data_tmp/data_jobkorea.csv",
    headless=False,
    limit=50,
    workers=5,
):
    opts = Options()
    if headless:
        opts.add_argument("--headless=new")
    opts.add_argument("--window-size=1400,900")
    opts.add_argument("--no-sandbox")
    opts.add_argument("--disable-dev-shm-usage")
    opts.add_argument(
        "user-agent=Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
        "AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36"
    )

    driver = webdriver.Chrome(options=opts)
    wait = WebDriverWait(driver, 20)

    try:
        driver.get("https://www.jobkorea.co.kr/")
        time.sleep(2)
        for _ in range(4):
            if close_popup(driver):
                break
            time.sleep(0.2)

        inp = wait.until(
            EC.element_to_be_clickable((By.CSS_SELECTOR, 'input[placeholder*="JOB 검색"], input[type="text"]'))
        )
        safe_click_elem(driver, inp)
        inp.send_keys(Keys.CONTROL, "a")
        inp.send_keys(Keys.BACKSPACE)
        inp.send_keys(keyword)
        time.sleep(0.2)
        close_popup(driver)

        try:
            inp.send_keys(Keys.ENTER)
            time.sleep(0.6)
        except Exception:
            pass

        try:
            wait.until(EC.presence_of_element_located((By.XPATH, "//a[contains(@href,'GI_Read')]")))
        except TimeoutException:
            driver.get(f"https://www.jobkorea.co.kr/Search/?stext={keyword}")
            time.sleep(1.5)

        cards = collect_cards(driver, limit=limit)
        if not cards:
            raise RuntimeError("카드를 찾지 못했습니다.")

        base_rows = []
        for card in cards:
            title, link = extract_title_link(card)
            base_rows.append({"title": title, "link": link})

    finally:
        driver.quit()

    print(f"상세 페이지 병렬 수집 중...")

    detail_map = {}

    with ThreadPoolExecutor(max_workers=workers) as executor:
        futures = {
            executor.submit(fetch_detail, row["link"]): row["link"]
            for row in base_rows
            if row["link"]
        }
        for future in as_completed(futures):
            link = futures[future]
            try:
                company, detail_list = future.result()
            except Exception:
                company, detail_list = "", []
            detail_map[link] = (company, detail_list)
            print(f"  완료: {link[:50]}...")

    rows = []
    for row in base_rows:
        company, detail_list = detail_map.get(row["link"], ("", []))
        rows.append(
            {
                "Site": "Job_Korea",
                "Col_Company": company,
                "Col_Recruit": row["title"],
                "Col_detail": detail_list,
                "Col_url": row["link"],
            }
        )

    df = pd.DataFrame(rows, columns=["Site", "Col_Company", "Col_Recruit", "Col_detail", "Col_url"])

    os.makedirs(os.path.dirname(out_path), exist_ok=True)

    df_to_save = df.copy()
    df_to_save["Col_detail"] = df_to_save["Col_detail"].apply(
        lambda x: json.dumps(x, ensure_ascii=False)
    )
    df_to_save.to_csv(out_path, index=False, encoding="utf-8-sig")

    return df


df_jobkorea = crawl_jobkorea_onepage_to_csv(
    keyword="데이터분석",
    out_path="data_tmp/data_jobkorea.csv",
    headless=False,
    limit=50,
    workers=5,
)

display(df_jobkorea)

상세 페이지 병렬 수집 중...
  완료: www.jobkorea.co.kr/Recruit/GI_Read/48535508?Oem_Co...
  완료: www.jobkorea.co.kr/Recruit/GI_Read/48617745?Oem_Co...
  완료: www.jobkorea.co.kr/Recruit/GI_Read/48569523?Oem_Co...
  완료: www.jobkorea.co.kr/Recruit/GI_Read/48617088?Oem_Co...
  완료: www.jobkorea.co.kr/Recruit/GI_Read/48381784?Oem_Co...
  완료: www.jobkorea.co.kr/Recruit/GI_Read/48607531?Oem_Co...
  완료: www.jobkorea.co.kr/Recruit/GI_Read/48603760?Oem_Co...
  완료: www.jobkorea.co.kr/Recruit/GI_Read/48535514?Oem_Co...
  완료: www.jobkorea.co.kr/Recruit/GI_Read/48565363?Oem_Co...
  완료: www.jobkorea.co.kr/Recruit/GI_Read/48598861?Oem_Co...
  완료: www.jobkorea.co.kr/Recruit/GI_Read/48591017?Oem_Co...
  완료: www.jobkorea.co.kr/Recruit/GI_Read/48516628?Oem_Co...
  완료: www.jobkorea.co.kr/Recruit/GI_Read/48580128?Oem_Co...
  완료: www.jobkorea.co.kr/Recruit/GI_Read/48580986?Oem_Co...
  완료: www.jobkorea.co.kr/Recruit/GI_Read/48561076?Oem_Co...
  완료: www.jobkorea.co.kr/Recruit/GI_Read/48623072?Oem_Co...
  완료: www.jobkorea.co.

,Site,Col_Company,Col_Recruit,Col_detail,Col_url
0,Job_Korea,㈜시오랩,"[강소기업] 보안솔루션 유지보수 엔지니어 모집(과장,차장)","[연금 보험 국민연금, 고용보험, 산재보험, 건강보험, 퇴직연금 휴무 휴가 행사 주...",www.jobkorea.co.kr/Recruit/GI_Read/48617088?Oe...
1,Job_Korea,㈜핀테크,[핀테크] 풀스텍 개발자 8년 이상 경력직을 모집합니다.,"[연금 보험 국민연금, 고용보험, 산재보험, 건강보험 휴무 휴가 행사 주5일제 편의...",www.jobkorea.co.kr/Recruit/GI_Read/48569523?Oe...
2,Job_Korea,㈜유니디아,OA 유지보수 필드(강북) 엔지니어 인력채용(자차보유 必),"[연금 보험 국민연금, 고용보험, 산재보험, 건강보험 휴무 휴가 행사 주5일제, 경...",www.jobkorea.co.kr/Recruit/GI_Read/48381784?Oe...
3,Job_Korea,넛지헬스케어㈜,[캐시워크] 데이터분석 담당 채용전환형 인턴,"[연금 보험 국민연금, 고용보험, 산재보험, 건강보험, 퇴직연금 휴무 휴가 행사 주...",www.jobkorea.co.kr/Recruit/GI_Read/48535508?Oe...
4,Job_Korea,콘센트릭스서비스코리아,[Catalyst] 데이터분석태깅/기획,"[신입·인턴, 정규직, 서울 강남구 테헤란로 509 (삼성동, 엔씨타워 I) NC타...",www.jobkorea.co.kr/Recruit/GI_Read/48617745?Oe...
5,Job_Korea,넛지헬스케어㈜,[캐시워크-병역특례] 데이터분석 담당 산업기능요원,"[연금 보험 국민연금, 고용보험, 산재보험, 건강보험 휴무 휴가 행사 주5일제, 연...",www.jobkorea.co.kr/Recruit/GI_Read/48535514?Oe...
6,Job_Korea,㈜인터엑스,[강남오피스]CAD데이터분석및자동화엔지니어,"[신입·인턴, 콘텐츠 마케터-팀장급(경력 8년 이상), 초대졸↑, 신입·경력, 서울...",www.jobkorea.co.kr/Recruit/GI_Read/48607531?Oe...
7,Job_Korea,㈜YG엔터테인먼트,[2월 수시채용] 데이터사이언스팀/데이터분석 및 예측모델링 담당자 (리더/경력),"[신입·인턴, 정규직, 서울특별시 마포구 희우정로1길 7 (합정동), 경력, 대졸이...",www.jobkorea.co.kr/Recruit/GI_Read/48603760?Oe...
8,Job_Korea,연이,데이터분석 준전문가 자격증 온라인 강의 교사 채용,"[신입·인턴, 프리랜서, 서울 광진구 아차산로36길 39 (자양동, 자양7차우성아파...",www.jobkorea.co.kr/Recruit/GI_Read/48565363?Oe...
9,Job_Korea,리딩스타㈜,[정규직] 리딩스타 경영본부 데이터분석 사업지원 담당 모집,"[신입·인턴, 정규직, 서울 송파구 삼전로 96 (삼전동), 경력, 경력무관, 대졸...",www.jobkorea.co.kr/Recruit/GI_Read/48598861?Oe...
